# IMPORTING

In [22]:
from pathlib import Path

import pandas as pd

# relative to the python folder
CSV_ROOT = Path("../.csv")
CURRENT = "2025-oct to 2026-oct"
LOOKBACK = "2024-oct to 2025-oct"

def load(folder, name):
    df = pd.read_csv(CSV_ROOT / folder / f"{name}.csv", dtype_backend="numpy_nullable")
    for col in df.columns:
        if col.endswith("_at"):
            df[col] = pd.to_datetime(df[col], utc=True)
    return df

def load_both(name):
    return pd.concat([load(LOOKBACK, name), load(CURRENT, name)], ignore_index=True)

prs = load_both("pull_requests")
issues = load_both("issues")
reviews = load_both("reviews")
review_comments = load_both("review_comments")
comments = load_both("comments")
# there might be overlap in users. drop dupes.
users = load_both("users").drop_duplicates("id", keep="last") 

# FILTERING

## FINDING BOTS

In [23]:
SOURCES = [
    (prs, "author", "author_type"),
    (issues, "author", "author_type"),
    (reviews, "reviewer", "reviewer_type"),
    (review_comments, "author", "author_type"),
    (comments, "author", "author_type"),
    (users, "login", "type"),
]

In [24]:
def suspect_bot(login_col: pd.Series, type_col) -> pd.Series:
    return (
        (type_col == "Bot")
        | login_col.str.contains("Bot", na=False, case=False)
    )

SUSPECTED_BOTS = set()
for df, login_col, type_col in SOURCES:
    sus = suspect_bot(df[login_col], df[type_col])
    SUSPECTED_BOTS.update(df.loc[sus, login_col].dropna())

for login in sorted(SUSPECTED_BOTS):
    try: print(users.loc[users["login"] == login, "url"].iloc[0])
    except: pass # sometimes it kind of just doesn't work

https://github.com/CodeBot-git
https://github.com/ComplexRobot
https://github.com/apps/copilot-pull-request-reviewer
https://github.com/Cube-bot-commit
https://github.com/DiplomaticRobot
https://github.com/DungenRobot
https://github.com/Errorbot1122
https://github.com/Hamood-bot
https://github.com/LevitatingRobot
https://github.com/Noobot9k
https://github.com/RobotBuddy
https://github.com/Roboticy3
https://github.com/RustyRoboticsBV
https://github.com/SaibotEssuluae
https://github.com/SleepingRobot
https://github.com/Warrobot10
https://github.com/WrobotGames
https://github.com/bamiebot-maker
https://github.com/benbot
https://github.com/bot2600
https://github.com/botero-dev
https://github.com/apps/dependabot
https://github.com/dusbot
https://github.com/ericbottard
https://github.com/jaredtalbot
https://github.com/mrbotue4
https://github.com/murraybot
https://github.com/tutabot
https://github.com/vehNbot


In [25]:
# After inspecting flagged accounts, only the following were determined as bots
BOTS = {
    "copilot-pull-request-reviewer[bot]",
    "dependabot[bot]",
    "Copilot",
}

## FINDING ALIASES

### Normalising the data

In [26]:
import re

def normalize_login(login: str):
    s = re.sub(r"[-_.]", "", login.lower())  # drop separators
    return re.sub(r"\d+$", "", s) or s       # drop trailing digits

def normalize_link(link: str):
    s = re.sub(r"^https?://", "", link.strip().lower())
    return re.sub(r"^www\.", "", s).rstrip("/")

humans = users[~users["login"].isin(BOTS)]

norm_users = pd.DataFrame({"login": humans["login"]})
norm_users["norm_login"] = humans["login"].map(normalize_login)
norm_users["norm_email"] = humans["email"].fillna("").str.strip().str.lower()
norm_users["norm_blog"] = humans["blog"].fillna("").map(normalize_link)

### Defining filtering methods

In [27]:
from difflib import SequenceMatcher

FILTER_LOGIN = True
FILTER_BLOG = True
FILTER_EMAIL = True

NAME_THRESHOLD = 0.85  # 0..1, how alike two names/logins must be
MIN_LENGTH = 4         # ignore very short names/logins, they match too easily

def is_similar(x: str, y: str):
    if len(x) < MIN_LENGTH or len(y) < MIN_LENGTH:
        return False
    m = SequenceMatcher(None, x, y)
    # the two quick ratios are cheap upper bounds that rule out most pairs before the slow exact ratio
    return m.real_quick_ratio() >= NAME_THRESHOLD and m.quick_ratio() >= NAME_THRESHOLD and m.ratio() >= NAME_THRESHOLD

# the 2 measures
def similar_name(a, b):
    return (FILTER_LOGIN and is_similar(a.norm_login, b.norm_login))

def same_contact(a, b):
    def match(x, y) -> bool: 
        return x != "" and x == y
    return (FILTER_BLOG and match(a.norm_blog, b.norm_blog)) or (FILTER_EMAIL and match(a.norm_email, b.norm_email))

# wrapper for the 2 measures
def is_alias(a, b):
    return similar_name(a, b) or same_contact(a, b)

### Filtering the data

In [28]:
from itertools import combinations

SUSPECTED_ALIASES = {}

def add_alias(a, b):
    main_a = SUSPECTED_ALIASES.get(a, a)
    main_b = SUSPECTED_ALIASES.get(b, b)
    if main_a == main_b: return  # already the same person

    # everything that pointed to main_a now points to main_b, then main_a itself does too
    for alt, main in SUSPECTED_ALIASES.items():
        if main == main_a:
            SUSPECTED_ALIASES[alt] = main_b
    SUSPECTED_ALIASES[main_a] = main_b

# populate SUSPECTED_ALIASES
for a, b in combinations(norm_users.itertuples(index=False), 2):
    if is_alias(a, b): add_alias(a.login, b.login)

for key, value in SUSPECTED_ALIASES.items():
    print(f"{users.loc[users['login'] == key, 'url'].iloc[0]} : {users.loc[users['login'] == value, 'url'].iloc[0]}")

https://github.com/aaronb97 : https://github.com/aaronp64
https://github.com/aarqon : https://github.com/aaronp64
https://github.com/aleks-313 : https://github.com/aleksa2808
https://github.com/Aleks59 : https://github.com/aleksa2808
https://github.com/AlexT-38 : https://github.com/alex44444444
https://github.com/Alex2782 : https://github.com/alex44444444
https://github.com/arlez80 : https://github.com/ARez2
https://github.com/aryan-11825114 : https://github.com/Ryan-000
https://github.com/brady444 : https://github.com/brad0
https://github.com/Carbonyte : https://github.com/cronyte
https://github.com/Castro1709 : https://github.com/Astro815
https://github.com/ccgargantua : https://github.com/gargantuan
https://github.com/Chenyang341 : https://github.com/yuchenyang1994
https://github.com/cjohnson57 : https://github.com/ecmjohnson
https://github.com/code32 : https://github.com/conde2
https://github.com/code807 : https://github.com/conde2
https://github.com/Dave9031 : https://github.com/D

In [29]:
# After inspecting flagged accounts, only the following were determined as aliases
ALIASES = {
    "UweSchneider" : "UweSchneiderGmbH"
}

## CLEANING THE DATASET

In [30]:
def clean(df, login_col):
    df = df[~df[login_col].isin(BOTS)].copy()                    # drop rows made by bots
    df["main_login"] = df[login_col].replace(ALIASES)
    df["main_login"] = df["main_login"].replace({"ghost": pd.NA})  # GITHUB marks *all different* deleted accounts as ghost 
    return df

prs = clean(prs, "author")
issues = clean(issues, "author")
reviews = clean(reviews, "reviewer")
review_comments = clean(review_comments, "author")
comments = clean(comments, "author")
users = clean(users, "login")

items = pd.concat([prs.assign(type="pull_request"), issues.assign(type="issue")], ignore_index=True)

# CONTRIBUTORS

## EVALUATING METRICS

In [32]:
START = pd.Timestamp("2025-10-18", tz="UTC")
END = pd.Timestamp("2026-10-18", tz="UTC")

def in_window(times):
    return (times >= START) & (times < END)

a_prs = prs[in_window(prs["created_at"])]
a_issues = issues[in_window(issues["created_at"])]
a_reviews = reviews[in_window(reviews["submitted_at"])]  # from both years: reviews on lookback-year PRs count too


In [33]:
contributors = pd.DataFrame({
    "prs_authored": a_prs.groupby("main_login").size(),
    "merged_prs_authored": a_prs[a_prs["merged"].fillna(False)].groupby("main_login").size(),
    "issues_authored": a_issues.groupby("main_login").size(),
    "prs_reviewed": a_reviews.groupby("main_login")["pr_number"].nunique(), # distinct pr-s
}).fillna(0).astype(int)

activity_cols = ["prs_authored", "issues_authored", "prs_reviewed"]
contributors["total_activity"] = contributors[activity_cols].sum(axis=1)
contributors["activity_types"] = (contributors[activity_cols] > 0).sum(axis=1)


In [34]:
def activities(df, time_col, authored):
    return pd.DataFrame({"main_login": df["main_login"], "at": df[time_col], "authored": authored})

activity = pd.concat([
    activities(a_prs, "created_at", True),
    activities(a_issues, "created_at", True),
    activities(a_reviews, "submitted_at", False),
])
activity["month"] = activity["at"].dt.strftime("%Y-%m")

per_login = activity.groupby("main_login")
contributors["active_months"] = per_login["month"].nunique()
contributors["last_activity"] = per_login["at"].max()
contributors["first_authored"] = activity[activity["authored"]].groupby("main_login")["at"].min()


In [35]:
lookback_prs = prs[prs["created_at"] < START]
lookback_issues = issues[issues["created_at"] < START]
prior_authors = set(lookback_prs["main_login"].dropna()) | set(lookback_issues["main_login"].dropna())

contributors["authored_in_lookback"] = contributors.index.isin(prior_authors)

## CATEGORIZING

In [38]:
KEY_PERCENTILE, KEY_MIN_MONTHS = 0.95, 6
FREQUENT_PERCENTILE, FREQUENT_MIN_MONTHS = 0.90, 3
NEWCOMER_DAYS = DORMANT_DAYS = 90

key_threshold = contributors["total_activity"].quantile(KEY_PERCENTILE, interpolation="higher")
frequent_threshold = contributors["total_activity"].quantile(FREQUENT_PERCENTILE, interpolation="higher")

c = contributors
c["is_key"] = (c["total_activity"] >= key_threshold) & (c["active_months"] >= KEY_MIN_MONTHS)
c["is_frequent"] = (c["total_activity"] >= frequent_threshold) & (c["active_months"] >= FREQUENT_MIN_MONTHS)
c["is_newcomer"] = (
    (c["first_authored"] >= END - pd.Timedelta(days=NEWCOMER_DAYS)) & ~c["authored_in_lookback"]
).fillna(False)  # people who only reviewed have no first_authored
c["is_dormant"] = c["last_activity"] < END - pd.Timedelta(days=DORMANT_DAYS)

contributors

,prs_authored,merged_prs_authored,issues_authored,prs_reviewed,total_activity,activity_types,active_months,last_activity,first_authored,authored_in_lookback,is_key,is_frequent,is_newcomer,is_dormant
main_login,,,,,,,,,,,,,,
014-code,3,2,0,3,6,2,1,2026-10-16 01:59:00+00:00,2026-10-05 09:23:04+00:00,False,False,False,True,False
0MAD03,0,0,1,0,1,1,1,2026-07-04 13:28:24+00:00,2026-07-04 13:28:24+00:00,False,False,False,False,True
0TheJCM0,0,0,0,1,1,1,1,2026-05-29 11:19:26+00:00,NaT,False,False,False,False,True
0ttoman,1,1,1,0,2,2,1,2026-02-21 14:49:32+00:00,2026-02-15 22:28:58+00:00,False,False,False,False,True
0x0ACB,1,0,4,0,5,2,3,2026-07-17 10:01:47+00:00,2026-01-12 17:45:34+00:00,True,False,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
zuedev,4,1,0,2,6,2,2,2026-03-09 23:49:52+00:00,2025-10-23 20:41:38+00:00,True,False,False,False,True
zuranthus,2,1,0,0,2,1,2,2026-07-13 22:40:25+00:00,2026-06-27 16:28:07+00:00,False,False,False,False,True
zvodd,1,0,0,1,2,2,1,2026-02-19 02:20:50+00:00,2026-02-18 02:22:06+00:00,False,False,False,False,True


# PROCESS-METRICS

In [39]:
# step 1: one category per person (key > frequent > occasional)
category = pd.Series("occasional", index=contributors.index)
category[contributors["is_frequent"]] = "frequent"
category[contributors["is_key"]] = "key"

# step 2: items in the window, each with its author's category
window_items = items[in_window(items["created_at"]) & items["main_login"].notna()].copy()
window_items["group"] = window_items["main_login"].map(category)

# step 3: each author's first item in the window
first_item = window_items.sort_values("created_at").groupby("main_login")["number"].first()

# step 4: the first item of someone who authored nothing in the lookback year gets its own group
is_first_timer = ~window_items["main_login"].isin(prior_authors)
window_items.loc[window_items["number"].isin(first_item) & is_first_timer, "group"] = "first-timer's 1st item"

window_items["group"].value_counts()

group
key                       6344
first-timer's 1st item    2661
occasional                2292
frequent                  1390
Name: count, dtype: int64

## 1. Time to First Response: [6] 
A measurement of how long it takes for another person to respond to an action that requires attention (Like a pull request or issue).

In [40]:
author_of = items.set_index("number")["main_login"]

# deleted accounts (main_login NA) count as someone else
review_by_author = (reviews["main_login"] == reviews["pr_number"].map(author_of)).fillna(False)
comment_by_author = (comments["main_login"] == comments["parent_number"].map(author_of)).fillna(False)

m1_reviews = reviews[~review_by_author]
m1_comments = comments[~comment_by_author]

In [41]:
m1_items = window_items.copy()

# time from creation until the first review / comment by another person
first_review_at = m1_reviews.groupby("pr_number")["submitted_at"].min()
first_comment_at = m1_comments.groupby("parent_number")["created_at"].min()
m1_items["first_review"] = m1_items["number"].map(first_review_at) - m1_items["created_at"]
m1_items["first_comment"] = m1_items["number"].map(first_comment_at) - m1_items["created_at"]

# the first response is whichever of the two came first (NaT = no response yet)
m1_items["first_response"] = m1_items[["first_review", "first_comment"]].min(axis=1)


In [42]:
LIMIT = pd.Timedelta(days=30)

# only items that had at least 30 days to get a response before the end of the window
m1_eligible = m1_items[m1_items["created_at"] <= END - LIMIT].copy()

print(f"eligible: {len(m1_eligible)} of {len(m1_items)} items")


eligible: 11687 of 12687 items


In [43]:
def censored_quantile(responses, q):
    # no response = slower than everything ("100 years"), so those items still count in the percentiles
    filled = responses.fillna(pd.Timedelta(days=36500))
    value = filled.quantile(q, interpolation="higher")  # "higher" picks a real value instead of averaging two
    return value / pd.Timedelta(hours=1) if value <= LIMIT else float("inf")  # inf = more than 30 days

def ttfr_summary(responses):
    return pd.Series({
        "n": len(responses),
        "% within 1 day": (responses <= pd.Timedelta(days=1)).mean() * 100,  # no response (NaT) counts as False
        "% within 30 days": (responses <= LIMIT).mean() * 100,
        "% never": responses.isna().mean() * 100,
        "median (h)": censored_quantile(responses, 0.50),
        "p75 (h)": censored_quantile(responses, 0.75),
        "p90 (h)": censored_quantile(responses, 0.90),
    })


In [44]:
# main view: PRs vs issues
m1_eligible.groupby("type")["first_response"].apply(ttfr_summary).unstack().round(1)


,n,% within 1 day,% within 30 days,% never,median (h),p75 (h),p90 (h)
type,,,,,,,
issue,4803.0,52.1,68.1,27.9,19.9,inf,inf
pull_request,6884.0,61.0,79.6,14.4,9.9,239.9,inf


In [45]:
# per author group
m1_eligible.groupby(["type", "group"])["first_response"].apply(ttfr_summary).unstack().round(1)

n  % within 1 day  % within 30 days  \
type         group                                                              
issue        first-timer's 1st item  1662.0            59.0              75.5   
             frequent                 594.0            51.9              68.2   
             key                     1361.0            43.0              58.0   
             occasional              1186.0            53.0              69.2   
pull_request first-timer's 1st item   776.0            57.7              72.0   
             frequent                 696.0            57.6              77.7   
             key                     4515.0            63.2              82.9   
             occasional               897.0            55.3              70.9   

                                     % never  median (h)  p75 (h)  p90 (h)  
type         group                                                          
issue        first-timer's 1st item     20.0        11.7    560.4      inf  
             frequent                   27.6        19.1      inf      inf  
             key                        38.9        61.6      inf      inf  
             occasional                 26.5        19.5      inf      inf  
pull_request first-timer's 1st item     24.7        10.6      inf      inf  
             frequent                   16.7        13.0    395.6      inf  
             key                        10.3         8.8    143.6      inf  
             occasional                 23.9        14.7      inf      inf

In [46]:
pd.set_option("display.max_colwidth", None)  # show full URLs

first_timer_items = m1_eligible[m1_eligible["group"] == "first-timer's 1st item"]
first_timer_items.nlargest(5, "first_response")[["url", "type", "first_response"]]

,url,type,first_response
22120,https://github.com/godotengine/godot/issues/114340,issue,293 days 05:03:19
8870,https://github.com/godotengine/godot/pull/114361,pull_request,287 days 14:07:55
21458,https://github.com/godotengine/godot/issues/112845,issue,279 days 08:38:36
22006,https://github.com/godotengine/godot/issues/114096,issue,275 days 11:48:16
22419,https://github.com/godotengine/godot/issues/115071,issue,267 days 02:13:19


In [48]:
never = first_timer_items[first_timer_items["first_response"].isna()]
never.sample(5, random_state=4)[["url", "type", "state", "closed_at"]]

,url,type,state,closed_at
23579,https://github.com/godotengine/godot/issues/117772,issue,open,NaT
25213,https://github.com/godotengine/godot/issues/121967,issue,open,NaT
12814,https://github.com/godotengine/godot/pull/121153,pull_request,open,NaT
21436,https://github.com/godotengine/godot/issues/112806,issue,closed,2025-11-15 10:21:41+00:00
12671,https://github.com/godotengine/godot/pull/120919,pull_request,open,NaT


## 2. Change request outcome

In [49]:
m2_prs = window_items[window_items["type"] == "pull_request"].copy()

# step 1: everything starts as still open
m2_prs["outcome"] = "still open"
# step 2: closed (merged PRs are closed too, so step 3 overwrites them)
m2_prs.loc[m2_prs["state"] == "closed", "outcome"] = "closed unmerged"
# step 3: merged
m2_prs.loc[m2_prs["merged"].fillna(False), "outcome"] = "merged"

m2_prs["outcome"].value_counts()

outcome
merged             4120
still open         2316
closed unmerged    1066
Name: count, dtype: int64

In [50]:
outcome_table = pd.crosstab(m2_prs["group"], m2_prs["outcome"], normalize="index") * 100
outcome_table["n"] = m2_prs["group"].value_counts()
outcome_table.round(1)

outcome,closed unmerged,merged,still open,n
group,,,,
first-timer's 1st item,40.2,20.8,39.0,853
frequent,17.0,42.6,40.4,759
key,7.1,68.6,24.3,4887
occasional,24.6,26.8,48.6,1003


## 3. Bus Factor

In [51]:
# reviews: one row per (reviewer, PR), without the author's own replies on their PR
window_reviews = reviews[in_window(reviews["submitted_at"]) & ~review_by_author]
reviewed = window_reviews.drop_duplicates(["main_login", "pr_number"])["main_login"].dropna()

# merges: who pressed the merge button on PRs merged in the window
merged_by = prs.loc[in_window(prs["merged_at"]), "merged_by"].dropna()

# comments: for comparison, a much more open activity
commented = comments.loc[in_window(comments["created_at"]), "main_login"].dropna()

In [52]:
def concentration(logins):
    counts = logins.value_counts()  # sorted, most active person first
    shares = counts / counts.sum()
    return pd.Series({
        "total": counts.sum(),
        "people": len(counts),
        "top person %": shares.iloc[0] * 100,
        "top 10 %": shares.head(10).sum() * 100,
        # absence factor: the fewest people that together do at least 50%
        "people for 50%": (shares.cumsum() < 0.5).sum() + 1,
    })

pd.DataFrame({
    "reviews (PRs reviewed)": concentration(reviewed),
    "merges": concentration(merged_by),
    "comments": concentration(commented),
}).T.round(1)


,total,people,top person %,top 10 %,people for 50%
reviews (PRs reviewed),9299.0,287.0,14.4,65.4,6.0
merges,4664.0,3.0,81.5,100.0,1.0
comments,44128.0,3962.0,10.6,36.6,31.0


In [ ]:
(reviewed.value_counts(normalize=True).head(10) * 100).round(1)

main_login
AThousandShips    14.4
akien-mga         10.1
KoBeWi             7.5
Calinou            6.5
bruvzg             6.5
YeldhamDev         5.3
Ivorforce          4.8
Repiteo            4.3
clayjohn           3.5
kitbdev            2.6
Name: proportion, dtype: Float64

In [54]:
(merged_by.value_counts(normalize=True) * 100).round(1)

merged_by
Repiteo      81.5
akien-mga    17.5
lawnjelly     1.0
Name: proportion, dtype: Float64

## 4. Conversion Rate

In [55]:
COHORT_END = END - pd.Timedelta(days=180)  # everyone in the cohort had at least 180 days to come back
RETURN_GAP = pd.Timedelta(days=30)         # activity within 30 days of the first item still counts as the first visit

m4 = window_items[
    (window_items["group"] == "first-timer's 1st item") & (window_items["created_at"] <= COHORT_END)
].copy()

print(f"cohort: {len(m4)} first-timers")


cohort: 1405 first-timers


In [56]:
# step 1: every PR, issue and review (not on their own PR) in the window
later = pd.concat([
    activities(a_prs, "created_at", True),
    activities(a_issues, "created_at", True),
    activities(window_reviews, "submitted_at", False),
])

# step 2: each person's last activity
last_seen = later.groupby("main_login")["at"].max()

# step 3: returned = active again at least 30 days after their first item
m4["returned"] = (m4["main_login"].map(last_seen) - m4["created_at"]) >= RETURN_GAP

print(f"returned: {m4['returned'].mean() * 100:.1f}%")

returned: 16.2%


In [57]:
def retention(df, by):
    return df.groupby(by)["returned"].agg(n="size", returned_pct=lambda r: r.mean() * 100).round(1)

In [58]:
# by type of first item
retention(m4, "type")

,n,returned_pct
type,,
issue,1017,14.3
pull_request,388,21.1


In [59]:
# by outcome of the first PR (see metric 2)
first_prs = m4[m4["type"] == "pull_request"]
retention(first_prs, first_prs["merged"].fillna(False).rename("first PR merged"))

,n,returned_pct
first PR merged,,
False,285,16.1
True,103,35.0


In [ ]:
# by whether the first item got a response within 30 days (see metric 1)
got_response = (m4["number"].map(m1_items.set_index("number")["first_response"]) <= LIMIT).rename("response within 30 days")
retention(m4, got_response)

,n,returned_pct
response within 30 days,,
False,363,15.2
True,1042,16.5


In [65]:
# by whether the first item got a response within 30 days (see metric 1)
got_response = (first_prs["number"].map(m1_items.set_index("number")["first_response"]) <= LIMIT).rename("response within 30 days PRs")
retention(first_prs, got_response)

,n,returned_pct
response within 30 days PRs,,
False,99,19.2
True,289,21.8
